# Does a model reason in the language you ask in?

The question we're chasing: when you pose a math problem in French, Russian, or Chinese, **what language does the model actually reason in?**

References / Inspiration

- [1] Shi, Freda, et al. "Language models are multilingual chain-of-thought reasoners, 2022."
- [2] Yong, Zheng-Xin, et al. "Crosslingual reasoning through test-time scaling." arXiv preprint arXiv:2505.05408 (2025).
- [3] Wendler, Chris, et al. "Do llamas work in english? on the latent language of multilingual transformers." Proceedings of the 62nd Annual Meeting of the Association for Computational Linguistics (Volume 1: Long Papers). 2024.
- [4] Muennighoff, Niklas, et al. "s1: Simple test-time scaling." Proceedings of the 2025 Conference on Empirical Methods in Natural Language Processing. 2025.
- [5] Fan, Zhiyuan, et al. "Toward Scalable Terminal Task Synthesis via Skill Graphs." arXiv preprint arXiv:2604.25727 (2026).


## Setup


In [1]:
%load_ext autoreload
%autoreload 2
import asyncio
from reasoning_demo.client import LLM
from reasoning_demo import mgsm as M
from reasoning_demo import traces as T   # loads the shipped s1.1 generations
from IPython.display import HTML, display

llm = LLM()
print('Model:', llm.model)
print('Languages:', ', '.join(M.LANGUAGES))

Model: meta-llama/llama-3.1-8b-instruct
Languages: bn, de, en, es, fr, ja, ru, sw, te, th, zh


## 0. Solving GSM8K

GSM8K is a math dataset. Let's have a look at a sample.


In [2]:
PROBLEM = 5   # which MGSM problem (0–249); the same index is the same problem in every language

prob_en = M.load_mgsm('en', n=PROBLEM + 1)[PROBLEM]
r_en = await M.asolve_direct(llm, prob_en.question, 'en')

print('REQUEST:')
print(r_en.detail['prompt'])
print('\nRESPONSE:')
print(r_en.text)
ok = 'correct' if M.is_correct(r_en.answer, prob_en.answer) else 'WRONG'
print(f'\n[extracted: {r_en.answer}  |  gold: {prob_en.answer}  |  {ok}]')

REQUEST:
Question: Kylar went to the store to buy glasses for his new apartment. One glass costs $5, but every second glass costs only 60% of the price. Kylar wants to buy 16 glasses. How much does he need to pay for them?
Answer:

RESPONSE:
## Step 1: Calculate the cost of the first glass
The cost of the first glass is $5.

## Step 2: Calculate the cost of the second glass
The second glass costs 60% of the price of the first glass, which is 0.6 * $5 = $3.

## Step 3: Determine the number of glasses that cost $5 and the number that cost $3
Kylar wants to buy 16 glasses. Since every second glass costs $3, we can divide 16 by 2 to find out how many glasses cost $5 and how many cost $3. 16 / 2 = 8, so 8 glasses cost $5 and 8 glasses cost $3.

## Step 4: Calculate the total cost of the glasses that cost $5
8 glasses cost $5 each, so the total cost of these glasses is 8 * $5 = $40.

## Step 5: Calculate the total cost of the glasses that cost $3
8 glasses cost $3 each, so the total cost of 

## 1. Solving MGSM

MGSM was composed by manually translating the 250 samples of GSM8K in ten different languages. Let's have a look at a couple of them.

Insights:

- The model responds in the language it was asked
- The model's performance degrades.


In [3]:
langs = ['fr', 'zh']   # French, Russian, Chinese
probs = {lang: M.load_mgsm(lang, n=PROBLEM + 1)[PROBLEM] for lang in langs}

# solve the same problem in each language, concurrently
results = await asyncio.gather(*(M.asolve_direct(llm, probs[l].question, l) for l in langs))

# print the full request (prompt sent) and response (model output) per language
for lang, r in zip(langs, results):
    name = M.LANGUAGES[lang]['name']
    ok = 'correct' if M.is_correct(r.answer, probs[lang].answer) else 'WRONG'
    print(f'========== {name} ({lang}) ==========')
    print('REQUEST:')
    print(r.detail['prompt'])
    print('\nRESPONSE:')
    print(r.text)
    print(f'\n[extracted: {r.answer}  |  gold: {probs[lang].answer}  |  {ok}]\n')

========== French (fr) ==========
REQUEST:
Question : Kylar se rend au magasin afin d'acheter des verres pour son nouvel appartement. Un verre coûte 5 $, mais chaque deuxième verre ne coûte que 60 % du prix. Kylar veut acheter 16 verres. Combien devra-t-il payer pour les acheter ?
Answer:

RESPONSE:
Kylar doit acheter 16 verres, mais le deuxième verre ne coûte que 60 % du prix. Cela signifie que le deuxième verre coûte 5 * 0,6 = $<<5*0,6=3>>3.
Le troisième verre coûte 60 % du prix du deuxième verre, soit 3 * 0,6 = $<<3*0,6=1,80>>1,80.
Le quatrième verre coûte 60 % du prix du troisième verre, soit 1,80 * 0,6 = $<<1,80*0,6=1,08>>1,08.
Le cinquième verre coûte 60 % du prix du quatrième verre, soit 1,08 * 0,6 = $<<1,08*0,6=0,648>>0,648.
Le sixième verre coûte 60 % du prix du cinquième verre, soit 0,648 * 0,6 = $<<0,648*0,6=0,3888>>0,3888.
Le septième verre coûte 60 % du prix du sixième verre, soit 0,3888 * 0,6 = $<<0,3888*0,6=0,23328>>0,23328.
Le huitième verre coûte 60 % du prix du septiè

## 2. Using a SOTA model

In this section we're using a specialized model which in math datasets competes and overcomes state-of-the-art models such as OpenAI's o1 and DeepSeek-R1.

Insights:

- The model no longer responds in the answer it was asked
- The model performs quite well in the foreign languages but not as good as in English


In [4]:
SIZE, BUDGET = '32B', 8000   # which s1.1 model + max thinking budget to load

# load the real s1.1 trace for the same PROBLEM in every language
s1_traces = {lang: T.load_trace(SIZE, lang, BUDGET, doc_id=PROBLEM) for lang in langs}

for lang in langs:
    tr = s1_traces[lang]
    ok = 'correct' if tr.correct else 'WRONG'
    print(f'========== {M.LANGUAGES[lang]["name"]} ({lang}) — s1.1-{SIZE} ==========')
    print('REQUEST:')
    print(M.build_prompt(tr.question, lang))
    print('\nRESPONSE (s1.1 reasoning):')
    print(tr.cot)
    print(f'\n[extracted: {tr.pred}  |  gold: {tr.gold}  |  {ok}]\n')

========== French (fr) — s1.1-32B ==========
REQUEST:
Question : Kylar se rend au magasin afin d'acheter des verres pour son nouvel appartement. Un verre coûte 5 $, mais chaque deuxième verre ne coûte que 60 % du prix. Kylar veut acheter 16 verres. Combien devra-t-il payer pour les acheter ?
Answer:

RESPONSE (s1.1 reasoning):
Okay, let's see. Kylar is going to the store to buy glasses for his new apartment. Each glass costs $5, but every second glass is 60% of the price. He wants to buy 16 glasses. How much does he need to pay?

Hmm, first, I need to make sure I understand the discount correctly. It says "chaque deuxième verre ne coûte que 60% du prix". So that means for every two glasses, the first one is full price ($5) and the second one is 60% of $5. So the discount applies to every alternate glass? Like, if he buys two glasses, the total would be $5 + (0.6 * $5) = $5 + $3 = $8. So for two glasses, it's $8. 

But he's buying 16 glasses. So since 16 is an even number, that means th

## 3. Inside look

Let's have a look at a particula trace where the model is asked the question in Russian and tries to respnd in english.

Insights

- The chain of thought is in English, but the final answer is finally translated in Russian
- The model burns reasoning budget on grammar not math (thinks about Russian, not in it)


In [5]:
tr = T.load_trace('32B', 'ru', budget=8000, doc_id=5)
print('Russian problem:', tr.question[:90], '...')
print('gold:', tr.gold, ' s1 answer:', tr.pred, ' correct:', tr.correct)
display(HTML(T.highlight_html(tr.cot, title='s1.1-32B reasoning on a Russian MGSM problem')))

Russian problem: Кайлар пошел в магазин, чтобы купить стаканы для своей новой квартиры. Один стакан стоит 5 ...
gold: 64  s1 answer: 64  correct: True


## 4. Using a reasoning model

Let's investigate a particular case of a reasoning model and specifically its **thinking** and **output** as two separate channels.

Insights

- The model "thinks" in English when it's prompted with the French and Russian problem but outputs its final solution in the prompted language.
- When the model is prompted in Chinese this shifts and the thinking is also done in Chinese.


In [6]:
MODELS = [
    "tencent/hy3-preview"
]
COMPARE_LANGS = ['fr', 'ru', 'zh']   # languages to compare
MAX_TOKENS = 4096                    # high enough for reasoning models to finish thinking

clients = {m: LLM(model=m) for m in MODELS}
qs = {l: M.load_mgsm(l, n=PROBLEM + 1)[PROBLEM] for l in COMPARE_LANGS}

# run every (language, model) pair concurrently; tolerate per-model failures
async def _run(lang, m):
    try:
        r = await M.asolve_direct(clients[m], qs[lang].question, lang, max_tokens=MAX_TOKENS)
        return lang, m, r, None
    except Exception as e:
        return lang, m, None, f'{type(e).__name__}: {e}'

pairs = await asyncio.gather(*(_run(l, m) for l in COMPARE_LANGS for m in MODELS))
grid = {(l, m): (r, err) for l, m, r, err in pairs}

# grouped first by language, then by model — print THINKING and OUTPUT text + their token counts
for lang in COMPARE_LANGS:
    name = M.LANGUAGES[lang]['name']
    print(f'################### {name} ({lang})   (gold: {qs[lang].answer}) ###################')
    print('REQUEST:')
    print(M.build_prompt(qs[lang].question, lang))
    print()
    for m in MODELS:
        r, err = grid[(lang, m)]
        print(f'----- {m} -----')
        if err:
            print(f'[ERROR: {err}]\n')
            continue
        u = r.usage
        thinking = r.detail.get('reasoning', '')
        output = r.detail.get('content', '') or r.text
        print(f'THINKING ({u.reasoning_tokens} tokens):')
        print(thinking if thinking else '[none exposed by this model]')
        print(f'\nOUTPUT ({u.output_tokens} tokens):')
        print(output if output else '[empty response]')
        ok = 'correct' if M.is_correct(r.answer, qs[lang].answer) else 'WRONG'
        print(f'\n[extracted: {r.answer}  |  {ok}]\n')
    print()

################### French (fr)   (gold: 64) ###################
REQUEST:
Question : Kylar se rend au magasin afin d'acheter des verres pour son nouvel appartement. Un verre coûte 5 $, mais chaque deuxième verre ne coûte que 60 % du prix. Kylar veut acheter 16 verres. Combien devra-t-il payer pour les acheter ?
Answer:

----- tencent/hy3-preview -----
THINKING (0 tokens):
We need to read the problem carefully. It says: "Kylar se rend au magasin afin d'acheter des verres pour son nouvel appartement. Un verre coûte 5 $, mais chaque deuxième verre ne coûte que 60 % du prix. Kylar veut acheter 16 verres. Combien devra-t-il payer pour les acheter ?"

Interpretation: There is a promotion: each second glass costs only 60% of the price. So the first glass is full price $5, the second glass is 60% of $5 = $3, the third glass is full price $5, the fourth is 60% of $5 = $3, and so on. So for every pair, the cost is $5 + $3 = $8. For 16 glasses, that's 8 pairs, so total = 8 * $8 = $64.

But we nee